# EthOn ontology pilot

This notebook loads the official EthOn source committed in this repository, checks its approved SHA-256 digest, and uploads those exact Turtle bytes to the local `ethon-pilot` dataset.

The dataset is in-memory. Re-running the preparation step is idempotent because it creates the dataset if needed and resets it with `DROP ALL` before upload. The inventory and smoke queries inspect explicit statements only; no inference or reasoner is enabled.

In [ ]:
import hashlib
import os
from pathlib import Path

from nl2sparql.kg.ontology.ethon_pilot import (
    inspect_ethon,
    load_ethon,
    load_smoke_queries,
)
from nl2sparql.kg.ontology.fuseki_pilot import FusekiPilotClient

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ETHON_PATH = ROOT / "data" / "ontologies" / "EthOn.ttl"
QUERY_PATH = (
    ROOT / "src" / "nl2sparql" / "kg" / "validation" / "ethon_smoke.sparql"
)

In [ ]:
graph = load_ethon(ETHON_PATH)
inventory = inspect_ethon(graph)
checksum = hashlib.sha256(ETHON_PATH.read_bytes()).hexdigest()
approved_checksum = "e73e19bf0d6bbb0e28b1497a73e4499ca78ee9c1e8c475fa31e7c821354ce71d"
assert checksum == approved_checksum

{
    "triples": len(graph),
    "classes": len(inventory.classes),
    "object_properties": len(inventory.object_properties),
    "datatype_properties": len(inventory.datatype_properties),
    "subclass_relations": len(inventory.subclass_relations),
    "checksum": checksum,
}

## Load the local Fuseki dataset

Connection settings come from environment variables. The client and credentials are deliberately not displayed.

In [ ]:
fuseki_url = os.environ.get("FUSEKI_URL", "http://localhost:3030")
username = os.environ.get("FUSEKI_ADMIN_USER", "admin")
password = os.environ.get("FUSEKI_ADMIN_PASSWORD", "admin")
client = FusekiPilotClient(fuseki_url, "ethon-pilot", username, password)
client.prepare_dataset()
client.upload_turtle(ETHON_PATH)

## Run the committed smoke queries

All three committed queries must return at least one binding. Only query indexes and row counts are displayed.

In [ ]:
queries = load_smoke_queries(QUERY_PATH)
assert len(queries) == 3
summaries = []
for index, sparql in enumerate(queries, start=1):
    result = client.query(sparql)
    try:
        bindings = result["results"]["bindings"]
    except (KeyError, TypeError) as error:
        raise AssertionError(f"Query {index} returned no results.bindings list") from error
    assert isinstance(bindings, list), f"Query {index} returned invalid bindings"
    assert bindings, f"Query {index} returned no bindings"
    summaries.append({"query": index, "rows": len(bindings)})

summaries